# Load Yelp JSON files into SQLite

In [2]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../yelp.db")

## Business

In [7]:
df_business = pd.read_json("../data/yelp_academic_dataset_business.json", lines=True)

# attributes and hours are dicts, SQLite can't store them -> convert to text
df_business['attributes'] = df_business['attributes'].astype(str)
df_business['hours'] = df_business['hours'].astype(str)

df_business.shape

(150346, 14)

In [8]:
df_business.to_sql("business", conn, if_exists="replace", index=False)

150346

## Tip and checkin

In [9]:
df_tip = pd.read_json("../data/yelp_academic_dataset_tip.json", lines=True)
df_tip.to_sql("tip", conn, if_exists="replace", index=False)
print("tip:", df_tip.shape)

df_checkin = pd.read_json("../data/yelp_academic_dataset_checkin.json", lines=True)
df_checkin.to_sql("checkin", conn, if_exists="replace", index=False)
print("checkin:", df_checkin.shape)

tip: (908915, 5)
checkin: (131930, 2)


## User

In [11]:
# user file is too big for RAM, load it in chunks
chunk_size = 100000
total_rows = 0

for i, chunk in enumerate(pd.read_json("../data/yelp_academic_dataset_user.json", lines=True, chunksize=chunk_size)):
    # lists -> text
    chunk['friends'] = chunk['friends'].astype(str)
    chunk['elite'] = chunk['elite'].astype(str)

    # first chunk creates the table, the others are appended
    if i == 0:
        chunk.to_sql("user", conn, if_exists="replace", index=False)
    else:
        chunk.to_sql("user", conn, if_exists="append", index=False)

    total_rows += len(chunk)
    print(f"Chunk {i+1} inserted, total rows: {total_rows}")

Chunk 1 inserted, total rows: 100000
Chunk 2 inserted, total rows: 200000
Chunk 3 inserted, total rows: 300000
Chunk 4 inserted, total rows: 400000
Chunk 5 inserted, total rows: 500000
Chunk 6 inserted, total rows: 600000
Chunk 7 inserted, total rows: 700000
Chunk 8 inserted, total rows: 800000
Chunk 9 inserted, total rows: 900000
Chunk 10 inserted, total rows: 1000000
Chunk 11 inserted, total rows: 1100000
Chunk 12 inserted, total rows: 1200000
Chunk 13 inserted, total rows: 1300000
Chunk 14 inserted, total rows: 1400000
Chunk 15 inserted, total rows: 1500000
Chunk 16 inserted, total rows: 1600000
Chunk 17 inserted, total rows: 1700000
Chunk 18 inserted, total rows: 1800000
Chunk 19 inserted, total rows: 1900000
Chunk 20 inserted, total rows: 1987897


## Review

In [12]:
# same for reviews (biggest file)
chunk_size = 100000
total_rows = 0

for i, chunk in enumerate(pd.read_json("../data/yelp_academic_dataset_review.json", lines=True, chunksize=chunk_size)):
    if i == 0:
        chunk.to_sql("review", conn, if_exists="replace", index=False)
    else:
        chunk.to_sql("review", conn, if_exists="append", index=False)

    total_rows += len(chunk)
    print(f"Chunk {i+1} inserted, total rows: {total_rows}")

Chunk 1 inserted, total rows: 100000
Chunk 2 inserted, total rows: 200000
Chunk 3 inserted, total rows: 300000
Chunk 4 inserted, total rows: 400000
Chunk 5 inserted, total rows: 500000
Chunk 6 inserted, total rows: 600000
Chunk 7 inserted, total rows: 700000
Chunk 8 inserted, total rows: 800000
Chunk 9 inserted, total rows: 900000
Chunk 10 inserted, total rows: 1000000
Chunk 11 inserted, total rows: 1100000
Chunk 12 inserted, total rows: 1200000
Chunk 13 inserted, total rows: 1300000
Chunk 14 inserted, total rows: 1400000
Chunk 15 inserted, total rows: 1500000
Chunk 16 inserted, total rows: 1600000
Chunk 17 inserted, total rows: 1700000
Chunk 18 inserted, total rows: 1800000
Chunk 19 inserted, total rows: 1900000
Chunk 20 inserted, total rows: 2000000
Chunk 21 inserted, total rows: 2100000
Chunk 22 inserted, total rows: 2200000
Chunk 23 inserted, total rows: 2300000
Chunk 24 inserted, total rows: 2400000
Chunk 25 inserted, total rows: 2500000
Chunk 26 inserted, total rows: 2600000
Chun

In [16]:
# reload checkin
df_checkin = pd.read_json("../data/yelp_academic_dataset_checkin.json", lines=True)
df_checkin.to_sql("checkin", conn, if_exists="replace", index=False)
print("checkin:", df_checkin.shape)

checkin: (131930, 2)


In [17]:
# check all tables are there
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(tables)

       name
0  business
1       tip
2      user
3    review
4   checkin
